In order to cache my LM's I use jupyter notebook

## Sections

Cell 1: Modules

Cell 2: Global vars and loggers

Cell 3: Helperfunctions

[Cell 4: LM builder](#cell-4-lm-builder)

[Cell 5: Helperclass](#cell-5-helper-class)

[cell 6: Main function](#cell-6-executing-function)

[cell 7: Build LMs](#cell-7-building-lms)

[cell 8: Executing main function](#cell-8-executing-main-function)

## Setting up...
Modules! (2 later in cell 7 to prevent circulair import errors)
and some more top-level statement code that shouldn't be toplevel

In [ ]:
from pathlib import Path
from pydub import AudioSegment
import copy
import time
from faster_whisper import WhisperModel
import evaluate
from speechbrain.inference.ASR import EncoderDecoderASR
from speechbrain.utils.fetching import LocalStrategy
import os
import logging
import soundfile as sf
import traceback 
import re
import whisperx
from nltk import ngrams
from nltk import word_tokenize
from nltk.lm import Laplace
from nltk.lm import MLE
from nltk.lm import NgramCounter
from num2words import num2words
from nltk.stem import SnowballStemmer
from nltk.stem.cistem import Cistem 
import warnings



Global variable that all code depends on... (I know its ugly coding)

In [ ]:
root = Path("InterCS.py").parent #__file__ does not work in jupyter enviroment for some reason.
logging.basicConfig(level=logging.CRITICAL)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
warnings.filterwarnings("ignore") #stop flooding of output by "UserWarning: batch_size (1) is >= total VAD chunks (1) ..."

(private helperfunctions that probably should not be toplevel statements... also prep should probably be part of the class, it doesnt work without it...)

In [ ]:
def getCSAudioNames():  
        #gets names and numbers from selected_files.txt as made by main.py
        sel_file_path = root / "input" / "selected_files.txt"
        CSfiles = dict()
        with open(sel_file_path, "r") as sf:
            for line in sf:
                l = line.split(' ')
                key = l[0][:-1] #slice off the ':'
                val = l[1].strip()
                CSfiles[key] = val
        return CSfiles

def getCSAudioPaths(CSnames):
        #gets rel paths(string) of list of files like "article\\audio.ogg"
        p = "input\\code-switched-audio\\german"
        CSpaths = list()
        for key, name in CSnames.items():
            pad = f"{p}\\{name}"
            CSpaths.append(pad)
        return CSpaths

def getTimeStampsAndSegmentText():
    #returns sorted dict by filenumb with a list of list of strings (CStimestamps)
    p = root / "input\\code-switching\\de-en\\text"
    csts_dict = dict()
    with open(p, "r", encoding="UTF-8") as f:
        for line in f:
             ls = line.split(' ')
             text = " ".join(ls[1:])
             _, key, start, end = ls[0].split('_')
             if(key in csts_dict):
                csts_dict[key].append([start, end, text[:-1]])
             else:
                csts_dict[key] = [[start, end, text[:-1]]] #-1 to remove \n
    csts_dict = dict(sorted(csts_dict.items())) #orders on keys
    return csts_dict

def prep(tekst, language="en", n2w=False):
    #necessary for CSFile, should move to that class.
    tekst = tekst.lower()
    #https://stackoverflow.com/questions/69734528/convert-number-inside-a-string-to-english-words inspiration of next line is: (answer on October 27th 2021 Scandav)
    tekst = re.sub(r'(\d+)', lambda m: num2words(m.group(), lang=language), tekst)
    return re.sub(r"[\d\,\.\'\"\(\)\[\]\:\;]", '', tekst)

def getWER(ref, hyp):
    eva = evaluate.load("wer")
    eva.add(reference=ref, prediction=hyp)
    wer = eva.compute()
    return wer

## Cell 4: LM builder<a class="anchor" id="cell8">

-with two OOV handlings (Laplace or Stupid backoff-setup NB:builder only allows for uni- and bigrams)
-two languages: german and english.
-stemming reduces 

In [ ]:
def buildLM(lan, context, OOV="sbo", stemming=True):
     #trainingtime:
     #(sbo, no stemming):1M >>> 350m, 300K > 350m (based on linear guess, complexity of ngramcounter is not linear), 100K = 45m(de)/..m(en), 10K ~30s
     #(sbo, stemming): 100K = ~2M Ngrams de:30m,en:10m, 300K = ~6M Ngrams en: (6370188 Ngrams, 133280.67m), de: (6707205 Ngrams, 129465.87m)
     #data from:
     #https://wortschatz.uni-leipzig.de/en/download/deu
     #https://wortschatz.uni-leipzig.de/en/download/eng
     tik = time.perf_counter()
     match context:
          case "10K":
             r = 10000
          case "100K":
             r = 100000
          case "300K":
             r= 300000
          case "1M":
             r= 1000000
     match lan:
          case "german" | "deutsch":
               p = root / "input" / f"deu_mixed-typical_2011_{context}" / f"deu_mixed-typical_2011_{context}-sentences.txt"
          case "english":
               p = root / "input" / f"eng_news-typical_2016_{context}" / f"eng_news-typical_2016_{context}-sentences.txt"
     ngc = NgramCounter()
     match OOV:
          case "sbo":
             lm = MLE(3, counter = ngc)
          case "lap":
             lm = Laplace(3, counter=ngc)
          case _:
             raise ValueError(f"OOV handling is not correctly specified. {OOV}") 
     lm.vocab.update(['</s>'])
     match lan:
            case "english":
             lan_code = "en"
            case "german" | "deutsch":
             lan_code = "de"
            case _:
             raise ValueError(f"{lan} not supported") 
     with open(p, 'r', encoding="UTF-8", errors="replace") as f:
         for i, line in enumerate(f):
             line = prep(line.split('\t')[1], language=lan_code, n2w=stemming)
             line = word_tokenize(line, language=lan)
             if(stemming):
               stemmer = Cistem() if lan=="german" else SnowballStemmer(lan) #snowball outperformed in german as shown by Weissweiler, Fraser (2017)
               line = [stemmer.stem(word) for word in line] #stemming in order to reduce vocab size, improve probability distribution and reduce runtime
             #unigrams, bigrams, trigrams to enable stupid back_off, higher-order n-gram increase training time. Short sentences might result in few true high-order ngrams (as opposed to false high-order ngrams with lots of padding)
             lm.vocab.update(line)
             if(OOV=="sbo"):
               ngc.update([ngrams(line, 1)]) 
               ngc.update([ngrams(line, 2, pad_right=True, right_pad_symbol='</s>', pad_left=True, left_pad_symbol='</s>')])
             ngc.update([ngrams(line, 3, pad_right=True, right_pad_symbol='</s>', pad_left=True, left_pad_symbol='</s>')])
             if(i%10000==0):
                 tak = time.perf_counter()
                 res_t = ((r-i)*((tak-tik)/(i+1)))/60
                 logger.info(f"{i/10000:0.0f}/{r/10000:0.0f} in total time: {(tak-tik)/60:0.2f}m. \n Total ngrams {ngc.N()}.")
                 logger.debug(line)
     tak = time.perf_counter()
     logger.info(f"Build LM{lan}{context}{OOV} stemming?{stemming} in {tak-tik/60:0.2f}m.")
     return lm

## Cell 5: Helper class <a class="anchor" id="cell8">

In [ ]:
class CSFile():
    models = dict()

    def __init__(self, article_name, ar_numb, path):
        self.name = article_name        #"art_name/audio{n}"
        self.article_numb = ar_numb     #000001
        self.full_path = path            #"root/art_name/audio{n}" this is actually full path...
        self.CS_ts = []                 #list of list[st, end, reftext, aud_path] (reftext not prepped)
        self.hyptexts = dict()          #dict[model_name] = list[hyptext_i], i per CSsegment in same ordering as CS_ts (=same index per CS_segment)
        self.number_of_CS = 0           #len(CS_ts) 
        self.CS_saver = dict()          #in order to save timestamps of a model, dict[modelname]:list() in ordering of CS_ts. (lazy workaround of having to rewrite core functions

    def getScores(self, model_name, gen=False):
        p = root / "outputs" / f"res_{model_name}_{context}_Stemmed.txt"
        model_nsl = model_name.split("_") #nsl = name, size, language_code
        lan = model_nsl[2]
        match lan:
               case "en":
                    ml = mlen
               case "de":
                    ml = mlge
               case _:
                    raise ValueError(f"{self.name}:{self.article_number} getScores() {lan} not recognised as language code.")
        if(gen):
          with open(p, "a", encoding="UTF-8") as f:
               for i in range(self.number_of_CS):
                    try:
                         hyp = self.hyptexts[model_name][i]
                    except Exception as e:
                         traceback.print_exc()
                         logger.warning(f"Error at {model_name}[{i}]. \n hyptexts: {self.hyptexts[model_name]} \n article number: {self.article_numb}, num of cs: {self.number_of_CS}")
                         hyp = "ERRORTOKEN"
                    stemmer = Cistem() if model_nsl[2]=="de" else SnowballStemmer("english") #Only stem with a single model for CS?  
                    hyp = word_tokenize(hyp)
                    hyp = [stemmer.stem(word) for word in hyp]
                    hyptri = ngrams(hyp, 3, pad_right=True, right_pad_symbol='</s>', pad_left=True, left_pad_symbol='</s>')#padding!
                    w = [] #not a dict, because what if same word twice in sentence?
                    ws = []
                    for tri in hyptri:
                        word = tri[2]
                        if(word!='</s>'):
                            sc = ml.unmasked_score(word, (tri[0], tri[1])) #eventhough very small values are unaestethic, logscore loses information due to (-inf), so use unmasked
                            if(sc==0): #stupid backoff implementation attempt, alpha is 0.4
                                sc = 0.4 * ml.unmasked_score(word, tri[1])
                                if(sc==0):
                                    sc = 0.4 * 0.4 * ml.unmasked_score(word)
                            w.append(f"({word}:{sc})")
                    w = " ".join(w)
                    f.write("%s_%s: %s \n" % (self.article_numb, i, w)) #article number_CS number: (word: score)_0...(word:score)_i
        #to-do:
        else:
          with open(p, "r", encoding="UTF-8") as f:
              pass #read already generator scores from file

    def addCSSegTimeStampsAndText(self, list):
        #list of list of start_time, end_time and text
        self.CS_ts = list
        if(list!=[]):
             self.number_of_CS = len(list)

    def addAudioSegs(self, new=False):
         #getaudioseg on path to folder
         self.segs=True
         if(new):
          aud = AudioSegment.from_file(self.full_path)
         for i in range(len(self.CS_ts)):
               st = self.CS_ts[i][0]
               end = self.CS_ts[i][1]
               name = self.name.split('/')
               out_p = root / f"input\\code-switched-audio\\german\\{name[0]}\\{name[1]}_seg_{st}-{end}.wav"
               self.CS_ts[i].append(out_p)
               if(new):
                    bud = copy.copy(aud)
                    st = float(st)
                    end = float(end)
                    seg = bud[st:end]
                    seg.export(out_f=out_p, format="wav")

    def testBeamSize(self,beam):
          #function to test results of varying the beamsize parameter of fasterwhisper, optimal seems 5, the default
          self._loadModel("small.en")
          r = self.number_of_CS
          p = root / "outputs" / "beamsizeEffect.txt"
         
          fw=CSFile.models["small.en"]
          with open(p, "a") as f:
                    ref = self.CS_ts[0][2]
                    aud_file_path = self.CS_ts[0][3]
                    f.write(ref)
                    for i in range(beam):
                        beam_size = i + 1
                        tik = time.perf_counter()
                        segments, info = fw.transcribe(aud_file_path, beam_size=beam_size, language="en", task="transcribe")
                        n=0
                        f.write("bs: %s\n" % (beam_size))
                        for segment in segments:
                              txt = segment.text
                              f.write("%s: hyp: %s\n" % (n, txt))
                              n=n+1
                        tak = time.perf_counter()
                        tim = tak-tik
                        logger.debug(f"Beamsize: {beam_size}, ({(tim):0.2f}s)")
                        f.write(f"{tim}s\n")

    def addHypText(self, model_name, gen=False):
          r = self.number_of_CS
          p = root / "outputs" / f"hyp_{model_name}.txt"
          self.hyptexts[model_name] = list()
          if(gen): #generates hyptext using model in paramater and transcribe method
               self._loadModel(model_name)
               with open(p, "a", encoding="UTF-8", errors="replace") as f:
                    for i in range(r):
                         aud_file_path = self.CS_ts[i][3]
                         hyptext = self._transcribe(model_name, aud_file_path)
                         f.write("%s_%s: %s \n" % (self.article_numb, i, hyptext)) #number of file_CSnumber: full hyptekst \n
                         if(self.hyptexts[model_name]):
                              self.hyptexts[model_name].append(hyptext)
                         else:
                              self.hyptexts[model_name] = [hyptext]
               logger.debug(f"Hyptekst of {self.name} generated using {model_name}")
          else: #get hyptexts from .txt-file if already generated and adds it to the CSfile
               with open(p, "r", encoding="UTF-8", errors="replace") as f: 
                    t = time.time()
                    i = 0
                    for line in f:
                         if(f"{self.article_numb}" not in line):
                              continue
                         hyptext = line.split(": ")[1]
                         if("[hyptekst_error]" in hyptext):
                              logger.debug(f"Read {hyptext} at {hyptext[0]} in hyp_{model_name}.txt: {self.name}_{self.article_numb}")
                         if(self.hyptexts[model_name]):
                              self.hyptexts[model_name].append(hyptext)
                         else:
                             self.hyptexts[model_name] = [hyptext]
                         i = i + 1
                         if(i==self.number_of_CS):
                             t2 = time.time()
                             logger.debug(f"Found {i}/{self.number_of_CS} hyptekst for {self.name} after {t2-t}s")
                             logger.debug(f"Saved {hyptext} in hyptexts[{model_name}]: {self.hyptexts[model_name]} at {i}")
                             break

    def _loadModel(self, model_name):
        model_nsl = model_name.split("_")
        model_n = model_nsl[0]
        match model_n:
          case "Umlaut":
               self._loadAsrUmlaut(model_nsl[2])
          case "fw":
               self._loadFasterWhisper(model_nsl[1], model_nsl[2])
          case "WX":
                self._loadWhisperX(model_nsl[1], model_nsl[2])
          case _:
               raise RuntimeError(f"Model name {model_name} not recognised")

    def _transcribe(self, model_name, audio):
       #https://github.com/AI-Natural-Language-Processing-Lab/whisperX-Whisper-Based-Automatic-Speech-Recognition/blob/main/README.md 
       #part of WhisperX code is from model card of whisperX
       #if timestamps are necessary, generator may be save in the dictionary
       try:
        model = CSFile.models[model_name]
        model_nsl = model_name.split("_")
        model_n = model_nsl[0]
        match model_n:
            case "Umlaut":
               audio = os.path.relpath(audio, root)
               return prep(model.transcribe_file(str(audio)), language="german")
            case "fw":
               segments, info = model.transcribe(audio, beam_size=5, language=model_nsl[2], task="transcribe")
               #preprocess
               txt = [segment.text for segment in segments]
               txtstr = " ".join(txt)
               return prep(txtstr, language=model_nsl[2])
            case "WX": 
               device="cuda"
               audioar = whisperx.load_audio(audio)
               result = model.transcribe(audioar, batch_size=1) #batch size 1 because there is only 1 VAD segment (audio file is very short, if >30s consider higher batch size) removes warning and no performance decrease 
               model_a, metadata = whisperx.load_align_model(language_code=model_nsl[2], device=device) #english model gets english language code in order to better align CS in English (hypothesis)
               result_aligned = whisperx.align(result["segments"], model_a, metadata, audioar, device)
               logger.debug(f"{model_name}: segments: {result_aligned["segments"]}") #dict with {start: ts, end: ts, text: txt, words: [{?}], logscore: ?} (logscore of what?, segment? probability of segment under what model?)
               txt = [segment["text"] for segment in result_aligned["segments"]]
               txtstr = " ".join(txt)
               self.CS_saver[model_name] = (result_aligned)     #save the generators to save the timestamps
               return prep(txtstr, language=model_nsl[2])
       except Exception as e:
           traceback.print_exc()
           logging.debug(f"Error in transcribe() by {model_name} of {self.name}.")
           return "[hyptekst_error]"

    def _loadFasterWhisper(self, size, language_code):
          #FasterWhisper is purely here to test code. Left over from early stages of methodology design
          #https://huggingface.co/distil-whisper/distil-large-v3
          #https://www.geeksforgeeks.org/python/python-check-given-key-already-exists-in-a-dictionary/
          if(f"fw{size}" in CSFile.models):
              return
          
          device = "cpu" #how to use cuda!!?
          compute_type = "int8"
          model_dir = root / "models" #param?
          
          model_size = size
          model = WhisperModel(model_size, compute_type=compute_type, device=device, language=language_code)
          CSFile.models[f"fw_{size}"] = model
          logger.info(f"fw_{size} loaded...")

    def _loadAsrUmlaut(self, language_code):
          #and https://huggingface.co/jfreiwa/asr-crdnn-german-umlaute
          model_name = f"Umlaut_U_{language_code}"
          if(model_name in CSFile.models):
               return
          
          CSFile.models[model_name] = EncoderDecoderASR.from_hparams(source="jfreiwa/asr-crdnn-german-umlaute", savedir="models/asr-crdnn-german-umlaute", local_strategy=LocalStrategy.COPY)
          logger.info(f"{model_name} loaded...")

    def _loadWhisperX(self, size, language_code):
          #https://github.com/m-bain/whisperX#accelerated-forced-alignment-optional is used for inspiration for this code
          if(f"WX_{size}_{language_code}" in CSFile.models):
              return
          #if device is GPU -> https://github.com/m-bain/whisperX#accelerated-forced-alignment-optional 30x increase faster for gpu 
          device = "cuda"
          CSFile.models[f"WX_{size}_{language_code}"] = whisperx.load_model(size, device=device, language=language_code, compute_type="float16", task="transcribe")
          logger.info
    
    def getGSText(self):
         #GS = golden standard = reference tekst
         list_of_text = [tu[2] for tu in self.CS_ts]
         return list_of_text

    def getTimeStamps(self):
         list_of_tuples = [tu[0:1] for tu in self.CS_ts]
         return list_of_tuples
    
    def __str__(self):
         return f"{self.name} ({self.article_numb})"

    def __eq__(self, value):
         if(value.name==self.name):
            return True
         else:
            return False

    def __len__(self):
         return self.number_of_CS

SyntaxError: unmatched ')' (2831807027.py, line 129)

## Cell 6: Executing function<a class="anchor" id="cell6">

In [ ]:
def createCSFiles(models, new=False, mode="r", start=0, r=0, beam=0, hyp=True, getScore=True): #models=list:tuple:(str: model_name, bool: gen)
     tik = time.perf_counter()
     CSnames = getCSAudioNames()
     CSpaths = getCSAudioPaths(CSnames)
     dictCS = getTimeStampsAndSegmentText()
     if(len(CSnames)!=len(CSpaths)):
          print("not same len")
          return
     
     CSval = list(CSnames.values())
     CSkey = list(CSnames.keys())
     
     listOfListsOfTS = list(dictCS.values())
     CSF_list = list()
     tok = time.perf_counter()
     logger.info(f"getting names, paths, timestamps and segmentstexts in {(tok-tik):0.3f}s")
     if(r==0):
          r = len(CSnames)
          if(start>0):
              r = r-start
     for i in range(r):
        n = i+start
        tok = time.perf_counter()
        p = CSpaths[i]
        CSF_list.append(CSFile(article_name= CSval[n], ar_numb= CSkey[n], path=p))
        CSF_list[i].addCSSegTimeStampsAndText(listOfListsOfTS[n])
        CSF_list[i].addAudioSegs(new)
        if(beam>0):
          CSF_list[i].testBeamSize(beam)
        if(hyp):
         for model_gen in models:
          CSF_list[i].addHypText(model_name=model_gen[0], gen=model_gen[1])
          if(getScore):
           CSF_list[i].getScores(model_name=model_gen[0], gen=True)
        tak = time.perf_counter()
        res_t = ((r-i)*((tak-tik)/(i+1)))/60
        logger.info(f"{i+1}/{r}: {(tak-tok):0.1f}s. Expected time left: {res_t:0.2f}m")
     logger.info(f"total time: {(tak-tik)/60:0.2f}m")
     return CSF_list

## Cell 7: Building LMs<a class="anchor" id="cell7">

In [16]:
logger.setLevel(logging.DEBUG)
print("starting...")
context = "10K" #training time increases ~nlogn (hypothesis) by size
logger.setLevel(logging.INFO)
mlge = buildLM("german", context, OOV="sbo", stemming=False)


INFO:__main__:0/1 in total time: 0.00m. 
 Total ngrams 18.


starting...


KeyboardInterrupt: 

In [18]:
context="100K" #76 m for 300K
mlen = buildLM("english", context, OOV="sbo", stemming=False)

INFO:__main__:0/10 in total time: 0.00m. 
 Total ngrams 15.
INFO:__main__:1/10 in total time: 0.80m. 
 Total ngrams 217527.
INFO:__main__:2/10 in total time: 2.18m. 
 Total ngrams 433863.
INFO:__main__:3/10 in total time: 3.82m. 
 Total ngrams 659793.
INFO:__main__:4/10 in total time: 6.14m. 
 Total ngrams 879504.
INFO:__main__:5/10 in total time: 8.34m. 
 Total ngrams 1103493.
INFO:__main__:6/10 in total time: 10.53m. 
 Total ngrams 1313244.
INFO:__main__:7/10 in total time: 13.13m. 
 Total ngrams 1539843.
INFO:__main__:8/10 in total time: 15.90m. 
 Total ngrams 1771377.
INFO:__main__:9/10 in total time: 19.32m. 
 Total ngrams 1983933.
INFO:__main__:Build LMenglish100Ksbo stemming?False in 126136.95m.


## Cell 8: Executing main function<a class="anchor" id="cell8">

In [ ]:
logger.setLevel(logging.INFO)

CSF_list = createCSFiles(models=[("WX_small.en_en", True)])


INFO:__main__:getting names, paths, timestamps and segmentstexts in 0.117s


2026-11-12 14:08:40 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint C:\Users\lucas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\whisperx\assets\pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint C:\Users\lucas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\whisperx\assets\pytorch_model.bin`
INFO:__main__:1/1: 250.8s. Expected time left: 4.18m
INFO:__main__:total time: 4.18m


In [ ]:
#descriptives
logger.setLevel(logging.INFO)
n = 0
i = 0
dur = 0.0
min=100000
max=0
avg=11.656045
sd = 0
for file in CSF_list:
    ni = len(file)
    n = n + ni
    i = i + 1
    if(ni<min):
        min = ni
    if(ni>max): 
        max = ni
print(n)
print(n/i) #11.656045
print(max)
print(min)
d = getTimeStampsAndSegmentText()
min = 10000000
max = 0
for key, value in d.items():
    dura = (float(value[0][1])-float(value[0][0]))
    if(dura<min):
        min = dura
    if(dura>max): 
        max = dura
    dur = dur + dura
print(dur/i)
print(max)
print(min)
#to do: WER!
#open each output file and get outputs per model into a dict: {'model':[hyp1,hyp2,hyp3]}
def getHyps(model):
    res = list()
    p = root / "outputs" / f"hyp_{model}.txt"
    with open(p, 'r') as f:
        for line in f:
            try:
                res.append(line)
            except:
                continue
    return res

hypdict = dict()
werdict = dict()
models = ["Umlaut_U_en", "WX_tiny.en_en", "WX_base.en_en", "WX_small.en_en", "WX_small.en_de", "WX_medium.en_de", "WX_small_en", "WX_small_de"]
for model in models:
    hypdict[model] = getHyps(model)
logger.info("Hypdict aangemaakt...")
refdict = getTimeStampsAndSegmentText().copy() #{article number:[[start, end, reftext],[start, end, reftext]]}
logger.info("Refdict aangemaakt...")
tik = time.perf_counter()
for model in models:
    i=0
    tot_wer=0.0
    for hyp in hypdict[model]:
     try:
        if("hyptekst_error" in hyp):
            continue
        a = hyp.split(": ")
        an = a[0].split("_")[0]
        ai = int(a[0].split("_")[1])
        ref = prep(refdict[an][ai][2])
        tot_wer = tot_wer + getWER(ref=ref, hyp=hyp)
        i = i + 1
        if(i%100==0):
            tak = time.perf_counter()
            logger.info(f"{model}: {i} wers in {tak-tik:0.2f}. {tot_wer:0.2f}")
     except:
        logger.info("Oh dan niet...")
        continue       
    werdict[model] = [tot_wer/i, tot_wer, i]
    with open(root / "outputs" / "WER", "a") as f:
        f.write(f"{model}: {tot_wer/i} ; {tot_wer}/{i}\n")
print("done")

NameError: name 'logger' is not defined

In [ ]:
for key, val in werdict.values():
    print(f"{key}:{val}")